In [ ]:
import logging
import boto3
import json
import pandas as pd
from botocore.exceptions import ClientError

Crea un notebook flights_analytics.ipynb en tu repositorio. El notebook debe:

Conectarse a la Read Replica con SQLAlchemy y pd.read_sql

Ejecutar las 8 preguntas (P1–P5 y W1–W3) como queries SQL

Mostrar el resultado de cada query como DataFrame de pandas

Incluir al menos una visualización por pregunta usando cualquiera de las siguientes librerías:

matplotlib — gráficas estáticas (plt.style.use('ggplot'))
plotnine — gramática de gráficas estilo ggplot2
plotly — gráficas interactivas
great_tables — tablas formateadas con resaltado, colores y barras inline
Elige la herramienta que mejor comunique la respuesta de cada pregunta: barras para rankings (P1, P2, P5), tablas formateadas para conteos (P3), líneas para series de tiempo (P4, W2), y tablas con ranking para W1 y W3.

In [ ]:
# ──────────────────────────────────────────────
# Configuración del logger
# ──────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
)
logger = logging.getLogger(__name__)

In [ ]:
def get_secret():

    secret_name = "itam/rds/flights/credentials"
    region_name = "us-east-1"

    # Create a Secrets Manager client
    session = boto3.session.Session()
    client = session.client(
        service_name='secretsmanager',
        region_name=region_name
    )

    try:
        get_secret_value_response = client.get_secret_value(
            SecretId=secret_name
        )
    except ClientError as e:
        logger.error(f"Error retrieving secret: {e}")
        raise e

    secret = get_secret_value_response['SecretString']
    return json.loads(secret)

creds   = get_secret()

In [ ]:
#conectarse a la base de datos con SQLAlchemy
from sqlalchemy import create_engine

# RDS endpoint Replica
RDS_ENDPOINT = "itam-flights-replica-956463123241.cmvsysimg4pq.us-east-1.rds.amazonaws.com"
DB_NAME      = creds["dbname"]
DB_USER      = creds["username"]
DB_PASSWORD  = creds["password"]
DB_PORT      = creds["port"]

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{RDS_ENDPOINT}:{DB_PORT}/{DB_NAME}"
)
logger.info("Conexión a la base de datos establecida con SQLAlchemy")

# Verificar conexión
try:
    with engine.connect() as conn:
        print("✓ Conexión exitosa a RDS PostgreSQL")
except SQLAlchemyError as e:
    print(f"✗ Error de conexión: {e}")

In [ ]:
# ──────────────────────────────────────────────
# Consultas de prueba
# ──────────────────────────────────────────────
query_P1 = """
SELECT origin_airport, destination_airport,
	   COUNT(*) AS numero_vuelos_por_ruta
FROM flights
GROUP BY origin_airport, destination_airport
ORDER BY numero_vuelos_por_ruta DESC
LIMIT 10;
"""

#Mostrar el resultado de cada query como DataFrame de pandas

with engine.connect() as conn:
    result_P1 = pd.read_sql(query_P1, conn)
    logger.info("✓ Consulta P1 ejecutada con éxito")
    display(result_P1)

In [ ]:
query_p2 = """
SELECT
    airline,
    COUNT(*) AS total_flights,
    SUM(CASE WHEN cancelled = 1 THEN 1 ELSE 0 END) AS cancelled_flights,
    SUM(CASE WHEN cancelled = 1 THEN 1 ELSE 0 END) * 1.0 / COUNT(*) AS avg_cancellations
FROM flights
GROUP BY airline
order by avg_cancellations desc
limit 10;
"""

with engine.connect() as conn:
    result_P2 = pd.read_sql(query_p2, conn)
    logger.info("✓ Consulta P2 ejecutada con éxito")
    display(result_P2)


In [ ]:
query_p3 = """
select 
	cancellation_reason,
	COUNT(*) as count_cancellation_number
from FLIGHTS
where cancellation_reason is not null
group by cancellation_reason
order by count_cancellation_number desc;"""

with engine.connect() as conn:
    result_P3 = pd.read_sql(query_p3, conn)
    logger.info("✓ Consulta P3 ejecutada con éxito")
    display(result_P3)

In [ ]:
query_p4 = """
select
	sum(departure_delay) as sum_departure_delay ,
	AVG(departure_delay) as avg_departure_delay,
	COUNT(*) as count_delay_per_month
from flights
WHERE cancelled = 0 AND departure_delay > 0
group by month;
"""

with engine.connect() as conn:
    result_P4 = pd.read_sql(query_p4, conn)
    logger.info("✓ Consulta P4 ejecutada con éxito")
    display(result_P4)

In [ ]:
query_p5 = """
select
	origin_airport,
	sum(weather_delay) as sum_weather_delay
from flights 
where weather_delay is not null
group by origin_airport
order by sum_weather_delay desc
limit 10;"""

with engine.connect() as conn:
    result_P5 = pd.read_sql(query_p5, conn)
    logger.info("✓ Consulta P5 ejecutada con éxito")
    display(result_P5)